# Gender and Canon Formation in Top Novels

#### Soham Solanki
---

## Introduction

What does it mean for a novel to be "great"? The two datasets explored in this notebook offer two very different answers to that question. The **Goodreads Top 500** ranks novels by how widely they are held across 16,000+ member libraries. The **NYT Fiction Bestsellers** list tracks which novels appeared on a weekly commercial popularity chart from 1931 to 2020 — a proxy for mass-market success.

Both datasets encode choices made by people and institutions, and those choices have patterns. The central question that this notebook is surrounding: **How does gender shape which novels get recognized as canonical — and does it matter whether we measure canonicity through libraries or commercial bestseller lists?**

---
## 1. Setup & Data Loading/Cleaning

In [1]:
import pandas as pd
import numpy as np
import altair as alt

alt.data_transformers.disable_max_rows()

gr = pd.read_csv("https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/refs/heads/main/datasets/top-500-novels/top-500-novels-metadata_2025-01-11.csv")
nyt = pd.read_csv("https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv", sep="\t")

# Clean numeric fields with commas
gr["gr_num_ratings"] = gr["gr_num_ratings"].str.replace(",", "").astype(float)
gr["gr_num_reviews"] = gr["gr_num_reviews"].str.replace(",", "").astype(float)

# Derived columns
gr["pub_decade"] = (gr["pub_year"] // 10) * 10
nyt["decade"] = (nyt["year"] // 10) * 10

# Find overlapping books
nyt_titles_upper = set(nyt["title"].str.upper().str.strip())
gr["in_nyt"] = gr["title"].str.upper().str.strip().isin(nyt_titles_upper)

print(f"Goodreads Top 500: {gr.shape[0]} novels, {gr.shape[1]} columns")
print(f"NYT Bestsellers:   {nyt.shape[0]:,} weekly entries, {nyt['title_id'].nunique():,} unique titles")
print(f"Novels in both datasets: {gr['in_nyt'].sum()}")

Goodreads Top 500: 500 novels, 31 columns
NYT Bestsellers:   60,386 weekly entries, 7,431 unique titles
Novels in both datasets: 140


---
## 2. Dataset Overviews

In [2]:
gr.head(3)

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url,pub_decade,in_nyt
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,318,211,1.810748e+09,17220427.0,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...,1600,False
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,172,133,1.156132e+10,66462036.0,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN,1860,False
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,373,68,3.373178e+09,50566653.0,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN,1880,False


In [3]:
nyt.head(3)

,year,week,rank,title_id,title,author,decade
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping,1930
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche,1930
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck,1930


### 2a. Publication Year Distribution — Goodreads Top 500

The Goodreads dataset spans works from 1021 CE to 2015, but the distribution is far from uniform.

In [ ]:
gr_modern = gr[gr["pub_year"] >= 1700].copy()

pub_hist = alt.Chart(gr_modern).mark_bar(color="#4e79a7", opacity=0.85).encode(
    alt.X("pub_year:Q", bin=alt.Bin(step=20), title="Publication Year"),
    alt.Y("count():Q", title="Number of Novels"),
    tooltip=[alt.Tooltip("pub_year:Q", bin=alt.Bin(step=20), title="Year (bin)"),
             alt.Tooltip("count():Q", title="Novels")]
).properties(
    title="Goodreads Top 500: Publication Year Distribution (post-1700)",
    width=640, height=300
).configure_axis(grid=False).configure_view(strokeWidth=0)

pub_hist

alt.Chart(...)

The list is heavily weighted toward novels published after 1900, with a big surge in works from the 1980s–2000s. This makes sense: more recent books tend to have more library holdings and more Goodreads activity because they're more accessible. It also means this dataset is not a pure "timeless classics" list; contemporary popularity plays a big role in the rankings.

### 2b. NYT Bestsellers: Unique Titles Per Decade

The NYT list started in 1931, and the number of distinct titles appearing on it each decade tells us how the list's scope changed over time.

In [5]:
nyt_decade_counts = (
    nyt.groupby("decade")["title_id"]
    .nunique()
    .reset_index()
    .rename(columns={"title_id": "unique_titles"})
)

nyt_bar = alt.Chart(nyt_decade_counts).mark_bar(color="#e15759", opacity=0.85).encode(
    alt.X("decade:O", title="Decade"),
    alt.Y("unique_titles:Q", title="Unique Titles on List"),
    tooltip=["decade:O", alt.Tooltip("unique_titles:Q", title="Unique Titles")]
).properties(
    title="NYT Bestsellers: Unique Titles per Decade (1931–2020)",
    width=500, height=280
).configure_axis(grid=False).configure_view(strokeWidth=0)

nyt_bar

alt.Chart(...)

The number of unique titles on the NYT list grew hugely from the 1990s onward, more than doubling between the 1990s and 2010s. This partly reflects the list expanding to include more categories and weeks, and also the growing publishing industry. It's a reminder that comparing raw counts across decades would be misleading.

---
## 3. Gender in the Goodreads Canon

The Goodreads dataset includes `author_gender`, coded as `male` or `female`. (a binary categorization that we need to keep in mind)

### 3a. Overall Gender Breakdown

In [6]:
gender_counts = gr["author_gender"].value_counts().reset_index()
gender_counts.columns = ["gender", "count"]

gender_pie = alt.Chart(gender_counts).mark_arc(innerRadius=60).encode(
    alt.Theta("count:Q"),
    alt.Color("gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=alt.Legend(title="Author Gender")),
    tooltip=["gender:N", "count:Q"]
).properties(
    title="Author Gender in Goodreads Top 500",
    width=300, height=300
)

gender_pie

alt.Chart(...)

Male authors account for 355 of the 500 novels (71%), while female authors account for 145 (29%). The canon, as measured by library holdings, skews heavily male. But this headline number alone doesn't tell us whether female-authored books are rated differently, or whether readers engage with them differently.

### 3b. Do Readers Rate Female-Authored Books Differently?

Let's look at two Goodreads metrics side by side: **average rating** (quality signal) and **number of ratings** (popularity/reach signal), broken down by gender.

In [8]:
rating_box = alt.Chart(gr).mark_boxplot(extent="min-max", size=40).encode(
    alt.X("author_gender:N", title="Author Gender",
          axis=alt.Axis(labelAngle=0)),
    alt.Y("gr_avg_rating:Q", title="Goodreads Avg Rating",
          scale=alt.Scale(domain=[3.2, 4.8])),
    alt.Color("author_gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=None)
).properties(
    title="Avg Rating by Author Gender",
    width=260, height=300
)

num_ratings_box = alt.Chart(gr).mark_boxplot(extent="min-max", size=40).encode(
    alt.X("author_gender:N", title="Author Gender",
          axis=alt.Axis(labelAngle=0)),
    alt.Y("gr_num_ratings:Q", title="Number of Ratings (log scale)",
          scale=alt.Scale(type="log")),
    alt.Color("author_gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=None)
).properties(
    title="Number of Ratings by Author Gender",
    width=260, height=300
)

(rating_box | num_ratings_box).configure_axis(grid=False).configure_view(strokeWidth=0)

alt.HConcatChart(...)

This is one of the most striking findings in the dataset. Female-authored novels in the top 500 have a higher median Goodreads rating than male-authored novels, and they attract far more ratings on average: female authors' books are rated roughly twice as frequently. In other words: the institutional canon underrepresents female authors (71% vs 29%), but readers on Goodreads engage more with the female-authored books that do make it in. 

This raises an interesting question: is the library-holdings ranking system undervaluing books that readers love most?

### 3c. Rating vs. Popularity — A Scatter by Gender

Let's look at individual books to see if any patterns emerge when we plot rating against reader engagement.

In [9]:
scatter = alt.Chart(gr).mark_circle(opacity=0.65, size=55).encode(
    alt.X("gr_num_ratings:Q", title="Number of Goodreads Ratings (log scale)",
          scale=alt.Scale(type="log")),
    alt.Y("gr_avg_rating:Q", title="Average Goodreads Rating",
          scale=alt.Scale(domain=[3.2, 4.8])),
    alt.Color("author_gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=alt.Legend(title="Author Gender")),
    tooltip=[
        alt.Tooltip("title:N", title="Title"),
        alt.Tooltip("author:N", title="Author"),
        alt.Tooltip("gr_avg_rating:Q", title="Avg Rating", format=".2f"),
        alt.Tooltip("gr_num_ratings:Q", title="# Ratings", format=",.0f"),
        alt.Tooltip("pub_year:Q", title="Published")
    ]
).properties(
    title="Goodreads Rating vs. Popularity, by Author Gender",
    width=620, height=380
).configure_axis(grid=False).configure_view(strokeWidth=0)

scatter

alt.Chart(...)

The most-rated books (far right) include both male and female authors, but several of the orange dots (female authors) cluster at the high end of the popularity axis. The books with the most Goodreads engagement aren't necessarily the ones at the top of the library-holdings ranking: they're often more recent, genre-leaning novels.

---
## 4. Gender Over Time: How the Canon Has Changed

The uneven split tells us about the overall list, but it hides how things have shifted across publication decades. When did female authors start getting into the canon more, and have things been improving?

In [11]:
gr_post1800 = gr[gr["pub_year"] >= 1800].copy()

gender_decade = (
    gr_post1800.groupby(["pub_decade", "author_gender"])
    .size()
    .reset_index(name="count")
)

total_per_decade = gender_decade.groupby("pub_decade")["count"].sum().reset_index(name="total")
female_per_decade = gender_decade[gender_decade["author_gender"] == "female"].copy()
female_per_decade = female_per_decade.merge(total_per_decade, on="pub_decade")
female_per_decade["pct_female"] = female_per_decade["count"] / female_per_decade["total"] * 100

# Stacked bar
stacked = alt.Chart(gender_decade).mark_bar().encode(
    alt.X("pub_decade:O", title="Publication Decade"),
    alt.Y("count:Q", title="Number of Novels", stack="normalize",
          axis=alt.Axis(format="%")),
    alt.Color("author_gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=alt.Legend(title="Author Gender")),
    tooltip=["pub_decade:O", "author_gender:N", "count:Q"]
).properties(
    title="Gender Composition of Goodreads Top 500, by Publication Decade (post-1800)",
    width=650, height=320
).configure_axis(grid=False).configure_view(strokeWidth=0)

stacked

alt.Chart(...)

In [13]:
line = alt.Chart(female_per_decade).mark_line(
    color="#f28e2b", strokeWidth=2.5
).encode(
    alt.X("pub_decade:O", title="Publication Decade"),
    alt.Y("pct_female:Q", title="% Female-Authored Novels",
          scale=alt.Scale(domain=[0, 80]))
)

points = alt.Chart(female_per_decade).mark_circle(
    color="#f28e2b", size=70
).encode(
    alt.X("pub_decade:O"),
    alt.Y("pct_female:Q"),
    tooltip=[
        alt.Tooltip("pub_decade:O", title="Decade"),
        alt.Tooltip("pct_female:Q", title="% Female", format=".1f"),
        alt.Tooltip("count:Q", title="Female-authored novels")
    ]
)

rule = alt.Chart(pd.DataFrame({"y": [50]})).mark_rule(
    strokeDash=[4,4], color="gray", opacity=0.5
).encode(y="y:Q")

(line + points + rule).properties(
    title="% Female-Authored Novels in Goodreads Top 500, by Publication Decade",
    width=650, height=300
).configure_axis(grid=False).configure_view(strokeWidth=0)

alt.LayerChart(...)

The share of female-authored novels in the top 500 is not on a steady upward climb — it's volatile and decade-dependent. The 1810s see a surprisingly high female presence (like Jane Austen, Mary Shelley), then the mid-to-late 19th century sees a dip. The 1930s show a bump, and the 1970s see another surge. But even in the 2000s (the most represented decade overall), female authors still don't crack 50%. The dotted line marks 50/50, a bar the top 500 has never reached in any decade with large representation.

It's also worth noting that most of the pre-1800 novels (Don Quixote, Beowulf, etc.) have no gender recorded or are male-authored, which drags the overall average further down.

---
## 5. The Overlap: When Library Canon Meets the Bestseller List

140 books from the Goodreads top 500 also appeared on the NYT Bestsellers list. What distinguishes the books that achieved both kinds of recognition: institutional prestige and commercial success?

### 5a. Overlap Rate by Gender

In [14]:
overlap_gender = (
    gr.groupby(["author_gender", "in_nyt"])
    .size()
    .reset_index(name="count")
)
overlap_gender["in_nyt_label"] = overlap_gender["in_nyt"].map(
    {True: "Also on NYT List", False: "Goodreads Only"}
)

overlap_bar = alt.Chart(overlap_gender).mark_bar().encode(
    alt.X("author_gender:N", title="Author Gender", axis=alt.Axis(labelAngle=0)),
    alt.Y("count:Q", title="Number of Novels", stack="normalize",
          axis=alt.Axis(format="%")),
    alt.Color("in_nyt_label:N",
              scale=alt.Scale(domain=["Also on NYT List", "Goodreads Only"],
                              range=["#59a14f", "#bab0ac"]),
              legend=alt.Legend(title="Appears on NYT List?")),
    tooltip=["author_gender:N", "in_nyt_label:N", "count:Q"]
).properties(
    title="Share of Goodreads Top 500 That Also Appeared on NYT Bestsellers List",
    width=340, height=300
).configure_axis(grid=False).configure_view(strokeWidth=0)

overlap_bar

alt.Chart(...)

Female-authored books in the Goodreads top 500 have a slightly higher overlap rate with the NYT Bestsellers list than male-authored books. This is a subtle but meaningful finding: when female authors do break into the institutional canon, they're somewhat more likely to also have achieved commercial bestseller status, which could suggest a pattern where female authors need to prove commercial viability to climb library rankings.

### 5b. Does Being on the NYT List Correlate with a Higher Goodreads Rank?

In [15]:
rank_comparison = alt.Chart(gr).mark_boxplot(extent="min-max", size=40).encode(
    alt.X("in_nyt:N",
          title="Also on NYT Bestsellers List?",
          axis=alt.Axis(labelExpr="datum.value == 'true' ? 'Yes' : 'No'",
                        labelAngle=0)),
    alt.Y("top_500_rank:Q",
          title="Goodreads Top 500 Rank (lower = more prestigious)",
          scale=alt.Scale(reverse=True)),
    alt.Color("in_nyt:N",
              scale=alt.Scale(domain=[True, False], range=["#59a14f", "#bab0ac"]),
              legend=None)
).properties(
    title="Goodreads Rank: Books On vs. Off the NYT List",
    width=320, height=320
).configure_axis(grid=False).configure_view(strokeWidth=0)

rank_comparison

alt.Chart(...)

Books that appeared on both the Goodreads list and the NYT Bestsellers tend to have somewhat lower Goodreads ranks on average than books that appear only on Goodreads. The most-prestigious positions in the Goodreads library-holdings ranking tend to go to classic literary novels that predate the NYT list (which started in 1931), or  works that never competed on the American commercial bestseller market. 

### 5c. NYT Bestsellers Over Time: When Did Overlap Books Peak?

In [16]:
overlap_books = gr[gr["in_nyt"] == True][["title", "author_gender"]].copy()
overlap_books["title_upper"] = overlap_books["title"].str.upper().str.strip()

nyt_copy = nyt.copy()
nyt_copy["title_upper"] = nyt_copy["title"].str.upper().str.strip()

nyt_overlap = nyt_copy[nyt_copy["title_upper"].isin(set(overlap_books["title_upper"]))]
nyt_overlap = nyt_overlap.merge(overlap_books[["title_upper", "author_gender"]], on="title_upper", how="left")

first_appearance = (
    nyt_overlap.groupby(["title_upper", "author_gender"])["year"]
    .min()
    .reset_index()
    .rename(columns={"year": "first_nyt_year"})
)
first_appearance["first_nyt_decade"] = (first_appearance["first_nyt_year"] // 10) * 10

first_by_decade = (
    first_appearance.groupby(["first_nyt_decade", "author_gender"])
    .size()
    .reset_index(name="count")
)

overlap_time = alt.Chart(first_by_decade).mark_bar().encode(
    alt.X("first_nyt_decade:O", title="First NYT Appearance (Decade)"),
    alt.Y("count:Q", title="Overlap Books First Appearing"),
    alt.Color("author_gender:N",
              scale=alt.Scale(domain=["male", "female"], range=["#4e79a7", "#f28e2b"]),
              legend=alt.Legend(title="Author Gender")),
    tooltip=["first_nyt_decade:O", "author_gender:N", "count:Q"]
).properties(
    title="Overlap Books (Goodreads + NYT): First NYT Appearance by Decade & Gender",
    width=620, height=300
).configure_axis(grid=False).configure_view(strokeWidth=0)

overlap_time

alt.Chart(...)

Among the books that achieved both library prestige and NYT bestseller status, female-authored titles first appearing on the NYT list surged notably in the 1980s and 2000s. Before the 1970s, the overlap is almost entirely male-authored, reflecting both the male-dominated publishing landscape of mid-century America and the NYT list's own historical biases.

---
## 6. Genre and Language Gaps

What genres and languages dominate the Goodreads top 500?

### 6a. Genre Distribution

In [17]:
genre_counts = gr["genre"].value_counts().reset_index()
genre_counts.columns = ["genre", "count"]
# Label "na" more clearly
genre_counts["genre"] = genre_counts["genre"].replace("na", "Unclassified")

genre_bar = alt.Chart(genre_counts).mark_bar(color="#76b7b2").encode(
    alt.Y("genre:N", sort="-x", title="Genre"),
    alt.X("count:Q", title="Number of Novels"),
    tooltip=["genre:N", "count:Q"]
).properties(
    title="Goodreads Top 500: Genre Distribution",
    width=520, height=340
).configure_axis(grid=False).configure_view(strokeWidth=0)

genre_bar

alt.Chart(...)

**221 of 500 novels (44%) have no genre label at all** — "Unclassified." This is a major data quality gap that would make any genre-based analysis unreliable. Among the labeled books, history, fantasy, and romance are the top genres, but we should be cautious about drawing conclusions from only half the dataset.

### 6b. Original Language Distribution

In [18]:
lang_counts = gr["orig_lang"].value_counts().reset_index()
lang_counts.columns = ["language", "count"]

lang_bar = alt.Chart(lang_counts).mark_bar(color="#edc948").encode(
    alt.Y("language:N", sort="-x", title="Original Language"),
    alt.X("count:Q", title="Number of Novels"),
    tooltip=["language:N", "count:Q"]
).properties(
    title="Goodreads Top 500: Original Language of Novels",
    width=520, height=300
).configure_axis(grid=False).configure_view(strokeWidth=0)

lang_bar

alt.Chart(...)

**Takeaway:** English-language novels make up 430 of the 500 (86%). French (25) and German (14) are distant runners-up. This list reflects the makeup of OCLC's member libraries — predominantly in the US, UK, Canada, and Australia. Works in Japanese, Chinese, Arabic, Hindi, or Swahili are essentially invisible in this ranking, which should give us skepticism about calling this a truly "global" top 500.

---
## 7. Takeaways & Open Questions

### What I found

1. **The Goodreads canon is male-dominated (71/29), but readers engage more with female-authored books.** Female authors have higher average ratings and nearly 2x more Goodreads ratings despite having far less spots in the top 500. The metric used to build the list, library holdings, appears to underweight books that readers love most.

2. **Female representation in the canon is not on a steady upward trend.** It's lumpy and dependent on the time period, never reaching equality even in the most recent decades.

3. **The NYT and Goodreads lists are measuring related but distinct things.** The most prestigious books on the Goodreads list (low rank numbers) are often pre-NYT-list classics. The overlap books that achieved both recognition skew toward mid-century and contemporary fiction.

4. **Data quality limits deeper analysis.** Genre data is missing for 44% of books. The gender variable is binary and may misclassify some authors. The NYT list's structure changed several times over its history.

5. **The Goodreads top 500 is profoundly English.** 86% English-language. Any claims to global representativeness must be heavily qualified.

### Open questions for further research

- **Race and ethnicity**: This dataset has no author race/ethnicity field. A critical gap: what would the canon look like mapped against author identity beyond gender?
- **Publisher patterns**: Do certain publishers dominate? Is the NYT list a predictor of future library holdings?
- **Ratings over time**: Are newer books rated more highly simply because they have more reviews, creating recency bias?